In [1]:
!pip install pyarrow


In [2]:
import pandas as pd

In [3]:
import pickle

In [4]:
import seaborn as sns
import matplotlib.pyplot as plt

In [5]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.linear_model import Ridge

from sklearn.metrics import mean_squared_error, root_mean_squared_error

In [6]:
mean_squared_error

<function sklearn.metrics._regression.mean_squared_error(y_true, y_pred, *, sample_weight=None, multioutput='uniform_average')>

In [7]:
import os, mlflow, pathlib

mlflow.set_experiment("nyc-taxi-experiment")
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_registry_uri("sqlite:///mlflow.db")



In [8]:
pd.__version__

'2.3.3'

In [9]:
def read_dataframe(filename):

    df = pd.read_parquet(filename)

    df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)
    df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)

    return df

In [10]:
df_train = read_dataframe('./data/green_tripdata_2021-01.parquet')
df_val = read_dataframe('./data/green_tripdata_2021-02.parquet')


In [11]:
len(df_train), len(df_val)

(73908, 61921)

In [12]:
df_train['PU_DO'] = df_train['PULocationID'] + '_' + df_train['DOLocationID']
df_val['PU_DO'] = df_val['PULocationID'] + '_' + df_val['DOLocationID']

In [13]:
df_val

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,...,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,duration,PU_DO
0,2,2021-02-01 00:34:03,2021-02-01 00:51:58,N,1.0,130,205,5.0,3.66,14.00,...,10.00,0.0,None,0.3,25.30,1.0,1.0,0.00,17.916667,130_205
1,2,2021-02-01 00:04:00,2021-02-01 00:10:30,N,1.0,152,244,1.0,1.10,6.50,...,0.00,0.0,None,0.3,7.80,2.0,1.0,0.00,6.500000,152_244
2,2,2021-02-01 00:18:51,2021-02-01 00:34:06,N,1.0,152,48,1.0,4.93,16.50,...,0.00,0.0,None,0.3,20.55,2.0,1.0,2.75,15.250000,152_48
3,2,2021-02-01 00:53:27,2021-02-01 01:11:41,N,1.0,152,241,1.0,6.70,21.00,...,0.00,0.0,None,0.3,22.30,2.0,1.0,0.00,18.233333,152_241
4,2,2021-02-01 00:57:46,2021-02-01 01:06:44,N,1.0,75,42,1.0,1.89,8.50,...,2.45,0.0,None,0.3,12.25,1.0,1.0,0.00,8.966667,75_42
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64567,2,2021-02-28 22:19:00,2021-02-28 22:29:00,None,NaN,129,7,NaN,2.63,10.04,...,0.00,0.0,None,0.3,10.34,NaN,NaN,NaN,10.000000,129_7
64568,2,2021-02-28 23:18:00,2021-02-28 23:27:00,None,NaN,116,166,NaN,1.87,8.33,...,1.89,0.0,None,0.3,10.52,NaN,NaN,NaN,9.000000,116_166
64569,2,2021-02-28 23:44:00,2021-02-28 23:58:00,None,NaN,74,151,NaN,2.40,12.61,...,0.00,0.0,None,0.3,12.91,NaN,NaN,NaN,14.000000,74_151
64570,2,2021-02-28 23:07:00,2021-02-28 23:14:00,None,NaN,42,42,NaN,1.11,11.95,...,0.00,0.0,None,0.3,15.00,NaN,NaN,NaN,7.000000,42_42


In [14]:
categorical = ['PU_DO']   #  ['PULocationID', 'DOLocationID']
numerical = ['trip_distance']

dv = DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')

X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')

X_val = dv.transform(val_dicts)


In [15]:
target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

In [16]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_val)

#mean_squared_error(y_val, y_pred, squared=False)
root_mean_squared_error(y_val, y_pred)

7.7587152133919135

In [ ]:
mlflow.sklearn.autolog(log_datasets=False)

In [25]:
with open('models/lin_reg.bin', 'wb') as f_out:
    pickle.dump((dv, lr), f_out)

In [26]:
with mlflow.start_run():
    
    mlflow.set_tag("developer","eriton")
    
    mlflow.log_param("train-data-path", "./data/green_tripdata_2021-01.parquet")
    mlflow.log_param("valid-data-path", "./data/green_tripdata_2021-02.parquet")
    
    lr = LinearRegression()
    lr.fit(X_train, y_train)

    y_pred = lr.predict(X_val)

    rmse = root_mean_squared_error(y_val, y_pred)
    
    mlflow.log_metric("rmse", rmse)
    
    mlflow.log_artifact(local_path="models/lin_reg.bin", artifact_path="models_pickle")

In [17]:
import xgboost as xgb

In [18]:
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope

/workspaces/mlops-zoomcamp/.venv/lib/python3.12/site-packages/hyperopt/atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [19]:
train = xgb.DMatrix(X_train, label=y_train)
valid = xgb.DMatrix(X_val, label=y_val)

In [20]:
def objective(params):
    with mlflow.start_run():
        mlflow.set_tag("model", "xgboost")
        mlflow.log_params(params)
        booster = xgb.train(
            params=params,
            dtrain=train,
            num_boost_round=1000,
            evals=[(valid, 'validation')],
            early_stopping_rounds=50
        )
        y_pred = booster.predict(valid)
        rmse = root_mean_squared_error(y_val, y_pred)
        mlflow.log_metric("rmse", rmse)

    return {'loss': rmse, 'status': STATUS_OK}

In [ ]:
search_space = {
    'max_depth': scope.int(hp.quniform('max_depth', 4, 100, 1)),
    'learning_rate': hp.loguniform('learning_rate', -3, 0),
    'reg_alpha': hp.loguniform('reg_alpha', -5, -1),
    'reg_lambda': hp.loguniform('reg_lambda', -6, -1),
    'min_child_weight': hp.loguniform('min_child_weight', -1, 3),
    'objective': 'reg:linear',
    'seed': 42
}

best_result = fmin(
    fn=objective,
    space=search_space,
    algo=tpe.suggest,
    max_evals=50,
    trials=Trials()
)

  0%|          | 0/50 [00:00<?, ?trial/s, best loss=?]

/workspaces/mlops-zoomcamp/.venv/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [12:48:48] WARNING: /__w/xgboost/xgboost/src/objective/regression_obj.cu:277: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()



[0]	validation-rmse:10.39534                          
[1]	validation-rmse:9.11850                           
[2]	validation-rmse:8.24042                           
[3]	validation-rmse:7.65218                           
[4]	validation-rmse:7.26231                           
[5]	validation-rmse:7.00502                           
[6]	validation-rmse:6.83255                           
[7]	validation-rmse:6.71593                           
[8]	validation-rmse:6.63955                           
[9]	validation-rmse:6.58397                           
[10]	validation-rmse:6.54696                          
[11]	validation-rmse:6.51843                          
[12]	validation-rmse:6.49763                          
[13]	validation-rmse:6.48214                          
[14]	validation-rmse:6.46911                          
[15]	validation-rmse:6.45774                          
[16]	validation-rmse:6.45033                          
[17]	validation-rmse:6.44200                          
[18]	valid

/workspaces/mlops-zoomcamp/.venv/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [12:50:08] WARNING: /__w/xgboost/xgboost/src/objective/regression_obj.cu:277: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()



[0]	validation-rmse:6.89035                                                      
[1]	validation-rmse:6.69167                                                      
[2]	validation-rmse:6.65530                                                      
[3]	validation-rmse:6.64581                                                      
[4]	validation-rmse:6.64191                                                      
[5]	validation-rmse:6.63256                                                      
[6]	validation-rmse:6.63004                                                      
[7]	validation-rmse:6.62792                                                      
[8]	validation-rmse:6.61043                                                      
[9]	validation-rmse:6.60663                                                      
[10]	validation-rmse:6.60542                                                     
[11]	validation-rmse:6.60053                                                     
[12]	validation-

...

[0]	validation-rmse:11.12090                                                    
[1]	validation-rmse:10.22724                                                    
[2]	validation-rmse:9.50171                                                     
[3]	validation-rmse:8.91530                                                     
[4]	validation-rmse:8.44687                                                     
[5]	validation-rmse:8.07710                                                     
[6]	validation-rmse:7.78073                                                     
[7]	validation-rmse:7.54734                                                     
[8]	validation-rmse:7.36278                                                     
[9]	validation-rmse:7.21992                                                     
[10]	validation-rmse:7.10612                                                    
[11]	validation-rmse:7.01607                                                    
[12]	validation-rmse:6.94339

/workspaces/mlops-zoomcamp/.venv/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [12:59:47] WARNING: /__w/xgboost/xgboost/src/objective/regression_obj.cu:277: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()



[0]	validation-rmse:11.52883                                                    
[1]	validation-rmse:10.92123                                                    
[2]	validation-rmse:10.38283                                                    
[3]	validation-rmse:9.90743                                                     
[4]	validation-rmse:9.48590                                                     
[5]	validation-rmse:9.11816                                                     
[6]	validation-rmse:8.79407                                                     
[7]	validation-rmse:8.51208                                                     
[8]	validation-rmse:8.26709                                                     
[9]	validation-rmse:8.05342                                                     
[10]	validation-rmse:7.86693                                                    
[11]	validation-rmse:7.70525                                                    
[12]	validation-rmse:7.56575

/workspaces/mlops-zoomcamp/.venv/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [13:01:01] WARNING: /__w/xgboost/xgboost/src/objective/regression_obj.cu:277: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()



[0]	validation-rmse:11.59085                                                    
[1]	validation-rmse:11.03130                                                    
[2]	validation-rmse:10.52956                                                    
[3]	validation-rmse:10.08061                                                    
[4]	validation-rmse:9.68027                                                     
[5]	validation-rmse:9.32341                                                     
[6]	validation-rmse:9.00660                                                     
[7]	validation-rmse:8.72536                                                     
[8]	validation-rmse:8.47702                                                     
[9]	validation-rmse:8.25741                                                     
[10]	validation-rmse:8.06413                                                    
[11]	validation-rmse:7.89403                                                    
[12]	validation-rmse:7.74457